In [9]:
import sqlite3
import pandas as pd

# ============================================================
# DATABASE SETUP
# ============================================================

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Load CSV files
sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# Show columns so we can verify the schema
print("Sales columns:")
print(sales.columns.tolist())

print("\nCustomers columns:")
print(customers.columns.tolist())

print("\nProducts columns:")
print(products.columns.tolist())

# Load DataFrames into SQLite tables
sales.to_sql(
    "sales",
    conn,
    index=False,
    if_exists="replace"
)

customers.to_sql(
    "customers",
    conn,
    index=False,
    if_exists="replace"
)

products.to_sql(
    "products",
    conn,
    index=False,
    if_exists="replace"
)

print("\nDatabase ready with 3 tables: sales, customers, products")


# ============================================================
# HELPER FUNCTION
# ============================================================

def run(sql):
    return pd.read_sql_query(sql, conn)


# ============================================================
# TASK 01 — 20 SQL QUERIES
# ============================================================

queries = {

1: """
-- Q1: Show the first 10 rows of the sales table.

SELECT *
FROM sales
LIMIT 10;
""",

2: """
-- Q2: Count the total number of orders.

SELECT
    COUNT(*) AS total_orders
FROM sales;
""",

3: """
-- Q3: Show all orders with a quantity of 5.

SELECT *
FROM sales
WHERE quantity = 5;
""",

4: """
-- Q4: List all products in the Electronics category.

SELECT *
FROM products
WHERE category = 'Electronics';
""",

5: """
-- Q5: Show the 5 most expensive products by unit price.

SELECT *
FROM products
ORDER BY unit_price DESC
LIMIT 5;
""",

6: """
-- Q6: Count how many customers are in each region.

SELECT
    region,
    COUNT(*) AS customer_count
FROM customers
GROUP BY region
ORDER BY customer_count DESC;
""",

7: """
-- Q7: Find the total quantity sold for each product.

SELECT
    product_id,
    SUM(quantity) AS total_quantity_sold
FROM sales
GROUP BY product_id
ORDER BY total_quantity_sold DESC;
""",

8: """
-- Q8: Find the average unit price per category.

SELECT
    category,
    AVG(unit_price) AS average_unit_price
FROM products
GROUP BY category
ORDER BY average_unit_price DESC;
""",

9: """
-- Q9: Count orders per product,
-- showing only products with more than 50 orders.

SELECT
    product_id,
    COUNT(*) AS order_count
FROM sales
GROUP BY product_id
HAVING COUNT(*) > 50
ORDER BY order_count DESC;
""",

10: """
-- Q10: Find the highest and lowest unit price.

SELECT
    MAX(unit_price) AS highest_unit_price,
    MIN(unit_price) AS lowest_unit_price
FROM products;
""",

11: """
-- Q11: Show each order's product name and price.

SELECT
    s.order_id,
    p.product_name,
    p.unit_price
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id;
""",

12: """
-- Q12: Compute total revenue across all sales.

-- Revenue = quantity × unit_price

SELECT
    SUM(
        s.quantity * p.unit_price
    ) AS total_revenue
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id;
""",

13: """
-- Q13: Compute total revenue per region.

SELECT
    c.region,
    SUM(
        s.quantity * p.unit_price
    ) AS total_revenue
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id
JOIN customers AS c
    ON s.customer_id = c.customer_id
GROUP BY c.region
ORDER BY total_revenue DESC;
""",

14: """
-- Q14: Compute total revenue per product category.

SELECT
    p.category,
    SUM(
        s.quantity * p.unit_price
    ) AS total_revenue
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
""",

15: """
-- Q15: Count sales referencing a customer
-- who does not exist in the customers table.

SELECT
    COUNT(*) AS missing_customer_sales
FROM sales AS s
LEFT JOIN customers AS c
    ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
""",

16: """
-- Q16: Rank all products by total revenue.

WITH product_revenue AS (

    SELECT
        p.product_id,
        p.product_name,
        SUM(
            s.quantity * p.unit_price
        ) AS total_revenue

    FROM sales AS s

    JOIN products AS p
        ON s.product_id = p.product_id

    GROUP BY
        p.product_id,
        p.product_name
)

SELECT
    product_id,
    product_name,
    total_revenue,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank

FROM product_revenue

ORDER BY revenue_rank;
""",

17: """
-- Q17: Compute a running total of revenue by month.

-- IMPORTANT:
-- sales.csv uses "date", NOT "order_date".

WITH monthly_revenue AS (

    SELECT
        strftime('%Y-%m', s.date) AS month,

        SUM(
            s.quantity * p.unit_price
        ) AS monthly_revenue

    FROM sales AS s

    JOIN products AS p
        ON s.product_id = p.product_id

    GROUP BY
        strftime('%Y-%m', s.date)
)

SELECT
    month,
    monthly_revenue,

    SUM(monthly_revenue) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING
        AND CURRENT ROW
    ) AS running_total_revenue

FROM monthly_revenue

ORDER BY month;
""",

18: """
-- Q18: Find the top-selling product within each category.

WITH product_sales AS (

    SELECT
        p.category,
        p.product_id,
        p.product_name,

        SUM(
            s.quantity
        ) AS total_quantity_sold

    FROM sales AS s

    JOIN products AS p
        ON s.product_id = p.product_id

    GROUP BY
        p.category,
        p.product_id,
        p.product_name
),

ranked_products AS (

    SELECT
        category,
        product_id,
        product_name,
        total_quantity_sold,

        RANK() OVER (
            PARTITION BY category
            ORDER BY total_quantity_sold DESC
        ) AS category_rank

    FROM product_sales
)

SELECT
    category,
    product_id,
    product_name,
    total_quantity_sold

FROM ranked_products

WHERE category_rank = 1

ORDER BY category;
""",

19: """
-- Q19: Find products that sold above
-- the average product revenue.

WITH product_revenue AS (

    SELECT
        p.product_id,
        p.product_name,

        SUM(
            s.quantity * p.unit_price
        ) AS total_revenue

    FROM sales AS s

    JOIN products AS p
        ON s.product_id = p.product_id

    GROUP BY
        p.product_id,
        p.product_name
),

average_revenue AS (

    SELECT
        AVG(total_revenue) AS avg_product_revenue
    FROM product_revenue
)

SELECT
    pr.product_id,
    pr.product_name,
    pr.total_revenue

FROM product_revenue AS pr

CROSS JOIN average_revenue AS ar

WHERE pr.total_revenue > ar.avg_product_revenue

ORDER BY pr.total_revenue DESC;
""",

20: """
-- Q20: Compute each region's revenue
-- as a percentage of total revenue.

WITH region_revenue AS (

    SELECT
        c.region,

        SUM(
            s.quantity * p.unit_price
        ) AS region_revenue

    FROM sales AS s

    JOIN products AS p
        ON s.product_id = p.product_id

    JOIN customers AS c
        ON s.customer_id = c.customer_id

    GROUP BY c.region
)

SELECT
    region,
    region_revenue,

    ROUND(
        100.0 * region_revenue /
        SUM(region_revenue) OVER (),
        2
    ) AS revenue_percentage

FROM region_revenue

ORDER BY revenue_percentage DESC;
"""
}


# ============================================================
# RUN ONE QUERY ONLY
# ============================================================

# Change this number from 1 to 20
question_to_run = 1

print("\n" + "=" * 70)
print(f"RUNNING Q{question_to_run}")
print("=" * 70)

result = run(queries[question_to_run])

print(result.to_string(index=False))


# ============================================================
# CLOSE DATABASE
# ============================================================

conn.close()

Sales columns:
['order_id', 'date', 'customer_id', 'product_id', 'quantity']

Customers columns:
['customer_id', 'customer_name', 'region', 'signup_year']

Products columns:
['product_id', 'product_name', 'category', 'unit_price']

Database ready with 3 tables: sales, customers, products

RUNNING Q1
order_id       date customer_id product_id  quantity
  O00001 31-10-2024        C999       P019         2
  O00002 14-03-2024        C999       P014         1
  O00003 21-10-2024        C999       P004         1
  O00004 03-01-2024        C999       P003         3
  O00005 18-10-2024        C999       P018         4
  O00006 15-10-2024        C010       P006         2
  O00007 12-10-2024        C013       P016         5
  O00008 31-08-2024        C046       P003         2
  O00009 21-06-2024        C048       P006         5
  O00010 15-09-2024        C003       P006         2


# TASK 01 — SQL Queries

## Objective

Practice SQL from basic to advanced using the `sales`, `customers`, and `products` tables.

## Topics Covered

* `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`
* `COUNT()`, `SUM()`, `AVG()`, `MIN()


In [4]:
# TASK 02 — Side-by-side: Pandas vs SQL
# Analysis Question:
# What are the top 3 products by total revenue in each category?

import pandas as pd
import sqlite3

# 1. LOAD DATA

sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# 2. PANDAS VERSION

# Merge sales with customer information
full = sales.merge(
    customers,
    on="customer_id",
    how="left"
)

# Merge with product information
full = full.merge(
    products,
    on="product_id",
    how="left"
)

# Calculate revenue for every sale
full["revenue"] = full["quantity"] * full["unit_price"]


# Calculate total revenue for each product in each category
product_revenue = (
    full.groupby(
        ["category", "product_id", "product_name"],
        as_index=False
    )["revenue"]
    .sum()
    .rename(columns={"revenue": "total_revenue"})
)


# Sort products by revenue inside each category
product_revenue = product_revenue.sort_values(
    ["category", "total_revenue"],
    ascending=[True, False]
)


# Select top 3 products from each category
pandas_result = (
    product_revenue
    .groupby("category", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)


print("=" * 80)
print("PANDAS RESULT — TOP 3 PRODUCTS BY REVENUE IN EACH CATEGORY")
print("=" * 80)

print(pandas_result.to_string(index=False))

# 3. SQL VERSION

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")


# Convert DataFrames into SQL tables
sales.to_sql(
    "sales",
    conn,
    index=False,
    if_exists="replace"
)

customers.to_sql(
    "customers",
    conn,
    index=False,
    if_exists="replace"
)

products.to_sql(
    "products",
    conn,
    index=False,
    if_exists="replace"
)


# SQL query
sql_query = """
WITH product_revenue AS (

    SELECT
        p.category,
        p.product_id,
        p.product_name,

        SUM(
            s.quantity * p.unit_price
        ) AS total_revenue

    FROM sales AS s

    LEFT JOIN customers AS c
        ON s.customer_id = c.customer_id

    LEFT JOIN products AS p
        ON s.product_id = p.product_id

    GROUP BY
        p.category,
        p.product_id,
        p.product_name
),

ranked_products AS (

    SELECT
        category,
        product_id,
        product_name,
        total_revenue,

        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY total_revenue DESC
        ) AS revenue_rank

    FROM product_revenue
)

SELECT
    category,
    product_id,
    product_name,
    total_revenue

FROM ranked_products

WHERE revenue_rank <= 3

ORDER BY
    category,
    total_revenue DESC;
"""


# Execute SQL query
sql_result = pd.read_sql_query(
    sql_query,
    conn
)


print("\n" + "=" * 80)
print("SQL RESULT — TOP 3 PRODUCTS BY REVENUE IN EACH CATEGORY")
print("=" * 80)

print(sql_result.to_string(index=False))

# 4. COMPARE PANDAS AND SQL RESULTS

# Select the same columns
pandas_check = pandas_result[
    ["category", "product_id", "product_name", "total_revenue"]
].copy()

sql_check = sql_result[
    ["category", "product_id", "product_name", "total_revenue"]
].copy()


# Sort both results in exactly the same order
pandas_check = pandas_check.sort_values(
    ["category", "product_id"]
).reset_index(drop=True)

sql_check = sql_check.sort_values(
    ["category", "product_id"]
).reset_index(drop=True)


# Make sure revenue values have the same numeric type
pandas_check["total_revenue"] = pandas_check[
    "total_revenue"
].astype(float)

sql_check["total_revenue"] = sql_check[
    "total_revenue"
].astype(float)


# Compare the results
results_match = pandas_check.equals(sql_check)


print("\n" + "=" * 80)
print("FINAL COMPARISON")
print("=" * 80)

print("Do Pandas and SQL results match?", results_match)

if results_match:
    print("SUCCESS: Both approaches produced the same results.")
else:
    print("WARNING: Pandas and SQL results are different.")


# Close database connection
conn.close()

PANDAS RESULT — TOP 3 PRODUCTS BY REVENUE IN EACH CATEGORY
   category product_id  product_name  total_revenue
Accessories       P013      Backpack           7315
Accessories       P015   Phone Stand           2466
Accessories       P014  Water Bottle           2300
Electronics       P001        Laptop         100800
Electronics       P020        Tablet          54600
Electronics       P004       Monitor          44250
     Office       P010 Standing Desk          68800
     Office       P009         Chair          28080
     Office       P008     Desk Lamp           7120
 Stationery       P012       Pen Set           2115
 Stationery       P011      Notebook           1264

SQL RESULT — TOP 3 PRODUCTS BY REVENUE IN EACH CATEGORY
   category product_id  product_name  total_revenue
Accessories       P013      Backpack           7315
Accessories       P015   Phone Stand           2466
Accessories       P014  Water Bottle           2300
Electronics       P001        Laptop         100800


# TASK 02 — Pandas vs SQL

## Analysis Question
Find the **top 3 products by total revenue in each category**.

Revenue is calculated as:

**Revenue = Quantity × Unit Price**

## Pandas
Used `merge()`, `groupby()`, `sort_values()`, and `head(3)` to combine the tables, calculate revenue, and find the top 3 products in each category.

## SQL
Used `JOIN`, `GROUP BY`, and the `ROW_NUMBER()` window function with `PARTITION BY category` to calculate and rank product revenue.

## Result
Both Pandas and SQL produced the **same results**, confirmed using `pandas_check.equals(sql_check)`.

## Comparison
- **Pandas:** Easier for Python-based data cleaning, exploration, and ML.
- **SQL:** Better for querying and processing large datasets inside databases.
- **500 million rows:** SQL would be preferred because processing can happen inside the database without loading the entire dataset into Pandas.
- **Different strengths:** SQL is strong for joins, filtering, aggregation, and window functions; Pandas is strong for data manipulation, analysis, visualization, and ML preprocessing.

## Conclusion
Pandas and SQL can solve the same analysis, but the choice depends on the size of the data and the type of work being performed.

In [ ]:
# TASK 03 — SQL Window Functions Practice

import pandas as pd
import sqlite3

# 1. Load CSV files

sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# Convert date column to datetime
sales["date"] = pd.to_datetime(sales["date"])

# 2. Create SQLite database

conn = sqlite3.connect(":memory:")

sales.to_sql("sales", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")

# 3. Running Total of Revenue
# Revenue = quantity * unit_price
# Running total increases as we move through dates.

running_total_query = """
WITH daily_revenue AS (
    SELECT
        s.date,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales s
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY s.date
)

SELECT
    date,
    revenue,
    SUM(revenue) OVER (
        ORDER BY date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM daily_revenue
ORDER BY date;
"""

running_total = pd.read_sql_query(running_total_query, conn)

print("=" * 70)
print("1. RUNNING TOTAL OF REVENUE")
print("=" * 70)
print(running_total.to_string(index=False))

# 4. Rank Customers by Total Spending

customer_rank_query = """
WITH customer_spending AS (
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(s.quantity * p.unit_price) AS total_spending
    FROM sales s
    JOIN customers c
        ON s.customer_id = c.customer_id
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY
        c.customer_id,
        c.customer_name
)

SELECT
    customer_id,
    customer_name,
    total_spending,
    RANK() OVER (
        ORDER BY total_spending DESC
    ) AS spending_rank
FROM customer_spending
ORDER BY spending_rank
LIMIT 10;
"""

customer_rank = pd.read_sql_query(customer_rank_query, conn)

print("\n" + "=" * 70)
print("2. TOP 10 CUSTOMERS BY TOTAL SPENDING")
print("=" * 70)
print(customer_rank.to_string(index=False))


# 5. Rank Customers Within Each Region
# PARTITION BY region restarts the ranking for every region.

regional_rank_query = """
WITH customer_spending AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.region,
        SUM(s.quantity * p.unit_price) AS total_spending
    FROM sales s
    JOIN customers c
        ON s.customer_id = c.customer_id
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY
        c.customer_id,
        c.customer_name,
        c.region
)

SELECT
    customer_id,
    customer_name,
    region,
    total_spending,
    RANK() OVER (
        PARTITION BY region
        ORDER BY total_spending DESC
    ) AS regional_rank
FROM customer_spending
ORDER BY region, regional_rank;
"""

regional_rank = pd.read_sql_query(regional_rank_query, conn)

print("\n" + "=" * 70)
print("3. CUSTOMER RANKING WITHIN EACH REGION")
print("=" * 70)
print(regional_rank.to_string(index=False))

# 6. Monthly Revenue + Previous Month Revenue
# LAG() gets the value from the previous row.

monthly_revenue_query = """
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales s
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY strftime('%Y-%m', s.date)
)

SELECT
    month,
    revenue,
    LAG(revenue) OVER (
        ORDER BY month
    ) AS previous_month_revenue
FROM monthly_revenue
ORDER BY month;
"""

monthly_revenue = pd.read_sql_query(monthly_revenue_query, conn)

print("\n" + "=" * 70)
print("4. MONTHLY REVENUE VS PREVIOUS MONTH")
print("=" * 70)
print(monthly_revenue.to_string(index=False))

# 7. Month-over-Month Growth

# Formula:
# ((Current Month - Previous Month) / Previous Month) * 100

mom_growth_query = """
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales s
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY strftime('%Y-%m', s.date)
),

monthly_comparison AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY month
        ) AS previous_month_revenue
    FROM monthly_revenue
)

SELECT
    month,
    revenue,
    previous_month_revenue,
    ROUND(
        ((revenue - previous_month_revenue)
        / previous_month_revenue) * 100,
        2
    ) AS mom_growth_percent
FROM monthly_comparison
ORDER BY month;
"""

mom_growth = pd.read_sql_query(mom_growth_query, conn)

print("\n" + "=" * 70)
print("5. MONTH-OVER-MONTH REVENUE GROWTH")
print("=" * 70)
print(mom_growth.to_string(index=False))

# 8. Close database
conn.close()

print("\n" + "=" * 70)
print("TASK 03 COMPLETED")
print("=" * 70)

1. RUNNING TOTAL OF REVENUE
               date  revenue  running_total
2024-01-02 00:00:00     2654           2654
2024-01-03 00:00:00     3103           5757
2024-01-04 00:00:00       85           5842
2024-01-05 00:00:00      250           6092
2024-01-06 00:00:00     3745           9837
2024-01-07 00:00:00      870          10707
2024-01-08 00:00:00     2890          13597
2024-01-09 00:00:00     2263          15860
2024-01-10 00:00:00      920          16780
2024-01-11 00:00:00       45          16825
2024-01-12 00:00:00     3520          20345
2024-01-13 00:00:00     1225          21570
2024-01-14 00:00:00     2920          24490
2024-01-15 00:00:00      240          24730
2024-01-16 00:00:00     1725          26455
2024-01-17 00:00:00      496          26951
2024-01-19 00:00:00      250          27201
2024-01-20 00:00:00      365          27566
2024-01-21 00:00:00     1170          28736
2024-01-22 00:00:00       90          28826
2024-01-23 00:00:00       60          28886
2024

C:\Users\neel2\AppData\Local\Temp\ipykernel_15152\1727540917.py:15: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sales["date"] = pd.to_datetime(sales["date"])


# TASK 03 — Window Functions Practice

## Objective

Practice SQL window functions using sales, customers, and products data.

Revenue is calculated as:

**Revenue = Quantity × Unit Price**

## Window Functions Used

* **SUM() OVER:** Calculates the running total of revenue ordered by date.
* **RANK():** Ranks customers by their total spending.
* **PARTITION BY:** Restarts customer ranking separately for each region.
* **LAG():** Gets the previous month's revenue for comparison.
* **MoM Growth:** Calculates the percentage change in revenue from the previous month.

## RANK vs DENSE_RANK vs ROW_NUMBER

* **RANK():** Gives the same rank to tied values and skips the next rank.
* **DENSE_RANK():** Gives the same rank to tied values but does not skip ranks.
* **ROW_NUMBER():** Gives every row a unique number, even when values are tied.

This matters when deciding how ties should be handled.

## LAG and LEAD

**LAG()** looks at a previous row, which is useful for comparing the current month with the previous month.

**LEAD()** looks at a following row, which is useful when comparing the current row with a future row.

## Why Window Functions?

Window functions are often cleaner than manually creating previous rows, rankings, or cumulative calculations in Pandas because the calculation can be expressed directly in SQL while keeping the original rows.

## Result

The task successfully demonstrates:

1. Running revenue total.
2. Top 10 customer ranking.
3. Customer ranking within each region using `PARTITION BY`.
4. Monthly revenue and previous month's revenue using `LAG()`.
5. Month-over-month revenue growth using `LAG()`.

## Conclusion

Window functions are useful for rankings, cumulative calculations, comparisons with previous or next rows, and grouped analysis without collapsing the original result set.
